# Currently meant to run locally - adjust path as needed!!!!!

### 1st normalization then compression

### 10k Hz is performing better than 4k

In [32]:
import os
#test locally - PoC
# where it's running
current_dir = os.getcwd()
print(f"you are in: {current_dir}")

# folder:
audio_folder = "../raw_data/audio_and_txt_files/" 

if os.path.exists(audio_folder):
    files = [f for f in os.listdir(audio_folder) if f.endswith('.wav')]
    print(f"{len(files)} files .wav")
else:
    print("not found")

you are in: /home/totid/code/antonella-dm/projeto/notebooks
920 files .wav


In [33]:
import librosa
import numpy as np
import pandas as pd
from tqdm import tqdm #shows progress
import os
import sys
import librosa.util

# FEATURES

In [34]:
def extract_features_raw(y, sr):
    
    # Compression - apply Mu-Law compression (standard in audio to enhance low-level signals)
    # 'compress' peaks and amplify breathing details
    mu = 255
    y_compressed = np.sign(y) * np.log1p(mu * np.abs(y)) / np.log1p(mu)
    
    #
    y = y_compressed # y here is the wave sound signl
    
    #end of compression - this can be hidden for testing preprocessing

    
    feat = {}
     # TEMPORAL
    feat['rms_mean'] = float(np.mean(librosa.feature.rms(y=y)))
    feat['zcr_mean'] = float(np.mean(librosa.feature.zero_crossing_rate(y=y)))
    
    # SPECTRAL
    feat['centroid_mean'] = float(np.mean(librosa.feature.spectral_centroid(y=y, sr=sr)))
    feat['rolloff_mean'] = float(np.mean(librosa.feature.spectral_rolloff(y=y, sr=sr)))
    feat['flatness_mean'] = float(np.mean(librosa.feature.spectral_flatness(y=y)))
    feat['flux_mean'] = float(np.mean(librosa.onset.onset_strength(y=y, sr=sr)))
    
    # MFCC (12, ignore 0) 
    mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13) # 12 mfccs + other 6 = 18 total
    for i in range(1, 13):
        feat[f'mfcc_{i}'] = float(np.mean(mfccs[i]))
        
    return feat


# SANITY CHECK

In [35]:
# sanit xheck
print("sanit check")

# white noise
mock_sr = 10000 #instead of the std 22050Hz
mock_y = np.random.uniform(-1, 1, mock_sr * 6)

#
test_feat = extract_features_raw(mock_y, mock_sr)

# chekc
errors = []
if not isinstance(test_feat, dict): errors.append("must be a dict")
if len(test_feat) != 18: errors.append(f"Esperado 18 colunas, mas recebi {len(test_feat)}.")
if any(np.isnan(list(test_feat.values()))): errors.append("NaN in features")

if not errors:
    print("done - 18 vlid columns")
    # example
    print(f"MFCC_1 example: {test_feat.get('mfcc_1'):.4f}")
    print(f"RMS example: {test_feat.get('rms_mean'):.4f}")
else:
    print("errors:")
    for err in errors: print(f" - {err}")

sanit check
done - 18 vlid columns
MFCC_1 example: -2.2008
RMS example: 0.8349


# MAP

In [36]:
# raw_data audios - adjust path as needed
diagnosis_path = "../raw_data/patient_diagnosis.csv" 

if os.path.exists(diagnosis_path):
    # adjust names CSV
    df_diag = pd.read_csv(diagnosis_path, names=['patient_id', 'diagnosis'])
    
    # create a dict for ultra-fast lookup: { '101': 'COPD', ... }
    diag_map = dict(zip(df_diag['patient_id'].astype(str), df_diag['diagnosis']))
    print(f"mapping created for {len(diag_map)} patients.")
else:
    print(f"not found")
    diag_map = {}

mapping created for 126 patients.


In [37]:
# configuration
TARGET_SR = 10000 # instead of the std 20050 
CHUNK_DURATION = 6 # original was 6s
STEP_DURATION = 2  # it defines the 3s window (so we can overlap) - can be commented if we want not to have it. ig chunk is 4, then define as 2
all_results = []

# using all files
print(f"processing {len(files)} files...")

for f in tqdm(files):
    path = os.path.join(audio_folder, f)
    
    try:
        # PREPROCESSING
        
        # Load + Resample + Normalization? check
        # standardizes all audio to Target_SR and scales amplitude to [-1, 1]
        y, sr = librosa.load(path, sr=TARGET_SR)
        
        # Cleaning - trim
        # removes silent intervals from the beginning and end
        y, _ = librosa.effects.trim(y)

        # IDENTIFICATION (extract ID to define slicing)
        p_id = f.split('_')[0]
        diagnosis = diag_map.get(p_id, "Unknown")

#NEW BLOCK OF SLICING WITH OVERLAP - can be commented
        # dynamic slicing
        samples_per_chunk = CHUNK_DURATION * TARGET_SR
        
        # if COPD, no overlap
        if diagnosis == 'COPD':
            step_size = samples_per_chunk 
        else:
            step_size = STEP_DURATION * TARGET_SR
            
        chunks = []
        for start_idx in range(0, len(y) - samples_per_chunk + 1, step_size):
            end_idx = start_idx + samples_per_chunk
            chunk = y[start_idx : end_idx]
            chunks.append(chunk)
#END OF BLOCK WITH SLICING OVERLAP 

        #OR!!!!!!!!!!!!

#SIMPLE 6 s slicing   (can be commented or uncommentend)     
        # slicing (6s windows) -breaks the cleaned audio into fixed 6-second segments
        #samples_per_chunk = CHUNK_DURATION * TARGET_SR 
        #chunks = [y[i : i + samples_per_chunk] for i in range(0, len(y), samples_per_chunk) 
                  #if len(y[i : i + samples_per_chunk]) == samples_per_chunk]
#END of simple 6S SLICING

        # FEATURE EXTRACTION - this cell is getting too long - maybe adjust it later
        
        
        for i, chunk in enumerate(chunks):
            # extract 18 features from the 6s processed chunk
            features = extract_features_raw(chunk, TARGET_SR) # it calculates the mean for the 6s slice and return the 18 features
            
            # add metadata and labels
            features['patient_id'] = p_id
            features['chunk_id'] = i
            features['original_file'] = f
            
            # map diagnosis from our diag_map (fallback to 'unknown')
            features['diagnosis'] = diag_map.get(p_id, "Unknown")
            
            all_results.append(features)
            
    except Exception as e:
        print(f"error on{f}: {e}")

# final df
df_final = pd.DataFrame(all_results)

# check
print(f"\ndone - dataset w {len(df_final)} rows.")
print(f"unique diagnoses found: {df_final['diagnosis'].unique()}")

processing 920 files...


100%|█████████████████████████████████████████████████████████████████████████████████| 920/920 [02:00<00:00,  7.65it/s]


done - dataset w 3606 rows.
unique diagnoses found: ['COPD' 'URTI' 'Pneumonia' 'Healthy' 'Bronchiolitis' 'Bronchiectasis'
 'LRTI' 'Asthma']


In [38]:
display(df_final.head(15))

,rms_mean,zcr_mean,centroid_mean,rolloff_mean,flatness_mean,flux_mean,mfcc_1,mfcc_2,mfcc_3,mfcc_4,...,mfcc_7,mfcc_8,mfcc_9,mfcc_10,mfcc_11,mfcc_12,patient_id,chunk_id,original_file,diagnosis
0,0.739170,0.001539,101.449467,127.325543,0.000056,1.957479,141.853965,44.231221,31.555366,29.084454,...,15.377089,13.318531,9.304475,7.877657,7.429528,8.447693,223,0,223_1b1_Pr_sc_Meditron.wav,COPD
1,0.696589,0.001879,98.285502,113.049523,0.000059,1.518246,143.352698,39.783529,33.081844,32.168925,...,15.717680,15.663969,10.037776,7.375149,6.357885,8.668443,223,1,223_1b1_Pr_sc_Meditron.wav,COPD
2,0.670110,0.001899,94.122156,103.863215,0.000033,1.650601,143.955065,47.641450,33.539781,33.385107,...,14.671383,14.495760,10.442390,6.406775,5.649503,9.201296,223,2,223_1b1_Pr_sc_Meditron.wav,COPD
3,0.674821,0.001382,84.610496,86.525093,0.000015,1.759859,140.524004,48.073060,33.807260,31.479628,...,16.430892,14.911375,10.164741,6.695922,5.836122,8.794603,223,3,223_1b1_Pr_sc_Meditron.wav,COPD
4,0.634854,0.001432,85.512759,88.056144,0.000027,1.697848,133.707197,48.023790,28.651836,32.951737,...,18.016766,16.560348,10.037829,6.151356,4.874320,9.356361,223,4,223_1b1_Pr_sc_Meditron.wav,COPD
5,0.571487,0.002859,125.969932,157.905191,0.000055,0.964975,167.485904,45.296105,36.987994,26.189388,...,13.064530,10.442768,9.929374,8.123261,7.592710,6.627879,134,0,134_2b2_Al_mc_LittC2SE.wav,COPD
6,0.564276,0.003228,131.890763,162.746623,0.000064,1.031429,162.254489,48.858230,36.628669,26.685495,...,13.474112,10.298442,10.123339,8.388004,7.523492,6.796979,134,1,134_2b2_Al_mc_LittC2SE.wav,COPD
7,0.575702,0.003083,125.846198,155.008607,0.000041,0.955744,164.854102,46.713063,36.420202,26.525707,...,12.772074,10.840732,9.735220,8.081765,7.398275,7.011733,134,2,134_2b2_Al_mc_LittC2SE.wav,COPD
8,0.558850,0.038036,836.557297,2037.581104,0.016995,1.866030,79.391297,19.184271,28.636034,13.126595,...,12.418782,5.313176,6.581544,6.533768,7.356993,3.391712,170,0,170_1b4_Al_mc_AKGC417L.wav,COPD
9,0.545645,0.027431,741.905334,1774.571306,0.008428,1.557405,91.049872,13.793619,24.032822,14.712279,...,13.501519,6.298031,6.484623,4.859992,5.194889,4.138644,170,1,170_1b4_Al_mc_AKGC417L.wav,COPD


In [39]:
print(df_final.columns.tolist())

['rms_mean', 'zcr_mean', 'centroid_mean', 'rolloff_mean', 'flatness_mean', 'flux_mean', 'mfcc_1', 'mfcc_2', 'mfcc_3', 'mfcc_4', 'mfcc_5', 'mfcc_6', 'mfcc_7', 'mfcc_8', 'mfcc_9', 'mfcc_10', 'mfcc_11', 'mfcc_12', 'patient_id', 'chunk_id', 'original_file', 'diagnosis']


In [40]:
print(df_final['diagnosis'].value_counts().head())

diagnosis
COPD              2597
Pneumonia          296
Healthy            275
URTI               182
Bronchiectasis     128
Name: count, dtype: int64


In [44]:
# Save CSV to raw_data
df_final.to_csv("../raw_data/xgboost_features_6s_10kHz_compressed_after_normalization_w_overlapping.csv", index=False)
print("done")

done


In [50]:
# sound file

# do librosa after compressing